# Guitar Chord Lab 1：単独ギター音源のコード認識

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shogo-ishikawa/2026_3Q_M4Th4/blob/main/projects/guitar-chord-lab/notebooks/01_single_guitar_chords.ipynb)　[GitHubで見る](https://github.com/shogo-ishikawa/2026_3Q_M4Th4/blob/main/projects/guitar-chord-lab/notebooks/01_single_guitar_chords.ipynb)

ギターだけが鳴っている短い音源から、**時刻付きのコード候補**を表示します。上のセルから順に実行してください。

**流れ**：セットアップ → 入力 → 再生 → 特徴量（クロマ）→ コード推定 → 保存 → 設定を変えて試す

- 所要時間の目安：15分程度（CPUだけで動きます。GPUは不要です）
- 推定するのは「ギターが演奏しているコード」です。初版は12音それぞれのメジャー／マイナー（24種類）と、N（コードなし）を扱います。
- 表示する「類似度」は、テンプレートとの形の近さです。**正解する確率ではありません**。
- コードの推定は誤ることがあります。結果を聴いて確かめながら使う、耳コピ支援ツールです。

## 1. セットアップ

新しいColabでは、講義リポジトリをHTTPSで取得し、`projects/guitar-chord-lab/` の作品だけをインストールします。

| 項目 | 場所 |
| --- | --- |
| リポジトリの取得先 | `/content/2026_3Q_M4Th4` |
| 作業ディレクトリ（相対パスの基準） | `/content/2026_3Q_M4Th4/projects/guitar-chord-lab` |
| 結果の保存先 | `outputs/01_single_guitar/`（作業ディレクトリからの相対パス） |

- 表示されるコミットは、使ったコードの版です。結果と一緒に記録してください。
- `BRANCH`（ブランチ）や `COMMIT`（コミット）を変えて実行し直すと、その版にそろえます。コードの版が変わったときは、セッションの再起動を求める表示が出ます。
- Colabのファイルはランタイムを終了すると消えます。残したい結果はダウンロードするか、Googleドライブにコピーしてください。
- ローカルで使う場合は、作品フォルダで `pip install -e .` を実行してから、`notebooks/` フォルダでこのNotebookを開きます。


In [ ]:
# セットアップ（最初に実行します。BRANCH や COMMIT を変えたときも実行し直します）
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/shogo-ishikawa/2026_3Q_M4Th4.git"
BRANCH = "main"  # 作業ブランチの内容を試すときは、ブランチ名に書き換えます
COMMIT = ""      # 実験を再現するときは、記録したコミット（例: "1a2b3c4"）を入れます
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = Path("/content/2026_3Q_M4Th4")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    # 実行するたびに BRANCH（COMMIT があればそのコミット）の内容にそろえます
    git = ["git", "-C", str(REPO_DIR)]
    before = subprocess.run(git + ["rev-parse", "HEAD"], capture_output=True, text=True).stdout
    subprocess.run(git + ["fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(git + ["checkout", "-q", "--detach", COMMIT or "FETCH_HEAD"], check=True)
    after = subprocess.run(git + ["rev-parse", "HEAD"], capture_output=True, text=True).stdout
    if before != after and "guitar_chord_lab" in sys.modules:
        raise RuntimeError("コードの版が変わりました。「ランタイム」→「セッションを再起動」を選んでから、このセルから実行し直してください")
    PROJECT_DIR = REPO_DIR / "projects" / "guitar-chord-lab"
    # この作品だけをインストールします（追加機能なし）
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{PROJECT_DIR}"], check=True)
else:
    # ローカルでは、notebooks/ フォルダで開いたこのNotebookの1つ上を作品フォルダとします
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(PROJECT_DIR)  # 以降の相対パスは、作品フォルダ（projects/guitar-chord-lab）が基準です
sys.path.insert(0, str(PROJECT_DIR / "src"))  # インストール直後でも import できるようにします
commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("作業ディレクトリ:", Path.cwd())
print("使用しているコミット:", commit or "（Gitの管理外。配布ZIPなら release_info.json を参照）")


In [ ]:
# このNotebookで使う関数を読み込みます
%matplotlib inline
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display

import guitar_chord_lab as gcl
from guitar_chord_lab.audio import load_audio, prepare_for_analysis
from guitar_chord_lab.chords import PITCH_NAMES, chord_notes, make_chord_template
from guitar_chord_lab.evaluation import evaluate_segments, reference_for_clip
from guitar_chord_lab.plots import plot_analysis, plot_chromagram, plot_templates
from guitar_chord_lab.recognition import ChordSettings, compute_chroma, cosine_similarity, recognise_chords
from guitar_chord_lab.results import chroma_frame_lines, print_segments, save_analysis

OUTPUT_DIR = Path("outputs/01_single_guitar")
print("Guitar Chord Lab", gcl.__version__)

## 2. 入力する音源を選ぶ

`SOURCE` で入力を選びます。最初はそのまま実行してください。

| SOURCE | 内容 | 正解注釈 |
| --- | --- | --- |
| `"guitarset"`（既定） | 実際のギター録音（GuitarSet、約3 MB をダウンロード） | あり（リードシートのコード） |
| `"demo"` | プログラムで作ったギター風の音（合成デモ） | あり（生成時に分かっている） |
| `"upload"` | 自分の音源をアップロード（WAV・FLAC・OGG・MP3） | なし（評価は省略） |

- GuitarSet は CC BY 4.0 で公開されている研究用データです。取り出した音源は `data/raw/guitarset/` に保存され、Gitには追加されません。
- ダウンロードに失敗した場合は、自動で合成デモに切り替えます。合成デモの結果は動作確認用で、実際の録音での精度を示すものではありません。
- M4A・AACなどは読み込めません。`ffmpeg -i 入力.m4a 出力.wav` でWAVに変換してください。

In [ ]:
SOURCE = "guitarset"  # "guitarset"（実際のギター録音）、"demo"（合成デモ）、"upload"（自分の音源）
START_S = 0.0         # 切り出しを始める時刻 [秒]（元ファイルの先頭が0秒）
DURATION_S = 20.0     # 切り出す長さ [秒]。最初は10〜30秒程度にします


def use_demo():
    """合成デモを作り、(音源, 正解注釈) のパスを返します。"""
    from guitar_chord_lab.synth import write_demo_dataset

    write_demo_dataset("data/generated/demo")
    return "data/generated/demo/demo_a_guitar.wav", "data/generated/demo/demo_a_annotation.csv"


ANNOTATION_PATH = None  # 正解注釈CSV（ない場合は None）
if SOURCE == "guitarset":
    from guitar_chord_lab.guitarset import CITATION, EXAMPLE_TRACKS, prepare_track

    try:
        INPUT_PATH, ANNOTATION_PATH = prepare_track(EXAMPLE_TRACKS[0], "data/raw/guitarset")
        print("出典:", CITATION)
    except OSError as error:
        print("GuitarSet を取得できなかったため、合成デモを使います。原因:", error)
        INPUT_PATH, ANNOTATION_PATH = use_demo()
elif SOURCE == "demo":
    INPUT_PATH, ANNOTATION_PATH = use_demo()
elif SOURCE == "upload":
    from google.colab import files  # Colab でだけ使えます

    uploaded = files.upload()
    INPUT_PATH = list(uploaded)[0]
print("入力ファイル:", INPUT_PATH)
print("正解注釈:", ANNOTATION_PATH)

## 3. 読み込んで聴く

音源の一部を切り出して読み込みます。サンプリング周波数（1秒あたりの値の数）、チャンネル数、長さを確認し、再生してみましょう。

In [ ]:
clip = load_audio(INPUT_PATH, START_S, DURATION_S)
print(f"サンプリング周波数 {clip.sample_rate} Hz、{clip.channels} チャンネル、長さ {clip.duration_s:.2f} 秒")
print(f"元ファイルの {clip.offset_s:.2f} 秒目から切り出しました（結果の時刻はこの位置を0秒とします）")
peak = float(np.max(np.abs(clip.samples)))
if peak == 0:
    print("注意: 読み込んだ音はすべて0（完全な無音）です")
display(Audio(clip.samples, rate=clip.sample_rate, normalize=peak > 0))  # 完全な無音は音量をそろえずに再生（0で割らないため）

times = np.arange(clip.samples.shape[1]) / clip.sample_rate
fig, ax = plt.subplots(figsize=(11, 2.4), layout="constrained")
ax.plot(times, clip.samples[0], linewidth=0.4)
ax.set_xlabel("Time in clip [s]")
ax.set_ylabel("Amplitude")
ax.set_title("Waveform", loc="left")
plt.show()

## 4. クロマ特徴量を見る

- **音の高さと周波数**：弦が1秒間に振動する回数が周波数 [Hz] です。1オクターブ上がると周波数は2倍になります。
- **STFT（短時間フーリエ変換）**：音を短い時間ごとに区切り、それぞれにどの周波数がどれだけ含まれるかを調べる方法です。
- **CQT（定Q変換）**：周波数を半音ごとの間隔で調べる方法です。音楽の音階と対応させやすくなります。
- **クロマ特徴量**：オクターブの違いをまとめ、C、C#、D、…、B の12音それぞれの強さにしたものです。低いCも高いCも「C」に足し合わせます。
- **倍音**：ギターの1つの音には、その2倍、3倍、…の周波数の音（倍音）も含まれます。3倍は5度上、5倍は長3度上の音名になるため、クロマには構成音以外の音も少し現れます。

この教材では、22050 Hz に変換した音から約0.023秒ごとにクロマを計算します。

In [ ]:
settings = ChordSettings()  # 既定の設定（configs/analysis_default.toml と同じ値）
print(settings)

mono = prepare_for_analysis(clip, settings.sample_rate)  # モノラル・22050 Hz に変換
chroma, frame_times = compute_chroma(mono.samples[0], settings.sample_rate, settings)
print("クロマ特徴量の形:", chroma.shape, "（12音 × フレーム数）")
print(f"フレームの間隔: {settings.hop_length / settings.sample_rate:.4f} 秒")

frame = len(frame_times) // 2  # 真ん中あたりのフレームの12個の値を見てみます
print(f"{frame_times[frame]:.2f} 秒のクロマ（最大値を1にそろえた値）:")
for line in chroma_frame_lines(chroma[:, frame]):  # 無音のフレームは0で割らず、その旨を表示します
    print(line)

In [ ]:
analysis = recognise_chords(clip, settings)  # 解析をまとめて行います（次の章で中身を説明します）

fig, ax = plt.subplots(figsize=(11, 3.5), layout="constrained")
image = plot_chromagram(ax, analysis)
ax.set_xlabel("Time in clip [s]")
fig.colorbar(image, ax=ax, label="Relative strength")
plt.show()

## 5. コードのテンプレートと類似度

**コードの構成音**：メジャーコードは「根音・長3度（半音4つ上）・完全5度（半音7つ上）」、マイナーコードは「根音・短3度（半音3つ上）・完全5度」の3音です。例えば C は C・E・G、Am は A・C・E です。

**テンプレート**は、構成音を1、それ以外を0にした12個の値です。24種類のコードについて作ります。

In [ ]:
for label in ["C:maj", "A:min", "G:maj", "E:min"]:
    print(f"{label:6} 構成音 {chord_notes(label)}  テンプレート {make_chord_template(label).astype(int)}")
fig = plot_templates()
plt.show()

**類似度（コサイン類似度）**は、12個の値の並び方がどれだけ似ているかを0〜1で表します。値の大きさ（音量）ではなく「形」を比べます。

次のセルでは、C・E・G が強く、A も少し鳴っているクロマを例に、いくつかのコードとの類似度を計算します。

In [ ]:
example = np.zeros(12)
example[PITCH_NAMES.index("C")] = 1.0
example[PITCH_NAMES.index("E")] = 0.8
example[PITCH_NAMES.index("G")] = 0.9
example[PITCH_NAMES.index("A")] = 0.3  # 少しだけ A も鳴っている

for label in ["C:maj", "C:min", "A:min", "E:min", "F:maj"]:
    print(f"{label:6} 類似度 {cosine_similarity(example, make_chord_template(label)):.3f}")

## 6. コード候補を推定する

`recognise_chords` 関数は次の順に処理します（`src/guitar_chord_lab/recognition.py`）。

1. `segment_seconds`（既定0.5秒）ごとの区間に分け、区間内のクロマを平均する
2. 区間ごとに24種類のテンプレートとの類似度を計算し、`smoothing_segments`（既定3）個の区間で平均する（平滑化）
3. 判定する
   - 区間の音量が `silence_db`（既定 -50 dBFS）未満なら **N（コードなし）**。初版では無音の区間だけを N とします。
   - 第1候補の類似度が `min_score` 未満、または第1候補と第2候補の差が `min_margin` 未満なら **判定保留（X、図では ?）**
   - それ以外は第1候補のコード
4. 同じコード（または N）が続く区間を1つにまとめる。判定保留の区間は、区間ごとの候補と類似度を残すため、まとめずに並べます

N と判定保留は別のものです。正解のコードや区間の境界は、推定には一切使いません。

In [ ]:
print_segments(analysis)

In [ ]:
reference = None  # 正解注釈（切り出した範囲だけ、切り出し位置を0秒とする時刻に直したもの）
if ANNOTATION_PATH:
    reference = reference_for_clip(ANNOTATION_PATH, clip)  # 注釈の誤りや、音源より長い注釈もここで見つけます

figure = plot_analysis(analysis, Path(INPUT_PATH).name, reference)
plt.show()

図の見方：

- 上段：クロマグラム（明るいほどその音が強い。無音のフレームは黒）
- 中段：推定したコード（青＝判定、灰＝N、橙＝判定保留）と、正解注釈（緑）
- 下段：第1候補の類似度と、第1候補と第2候補の差。点線・破線は判定に使うしきい値

## 7. 正解注釈と比べる

正解注釈があれば、**正解と予測の区間が重なっている時間**で評価します。

- 正解率（accuracy）＝ 正解コードと一致した時間 ÷ 評価対象の時間
- 判定率（decided_ratio）＝ 判定保留でも予測なしでもない時間 ÷ 評価対象の時間

判定保留の時間は評価対象から除かず、不正解として数えます。

In [ ]:
metrics = None
if reference:
    metrics = evaluate_segments(reference, analysis.segments)
    print(f"正解率 accuracy      = {metrics['accuracy']:.3f}（正解 {metrics['correct_s']:.2f}秒 / 評価対象 {metrics['evaluated_s']:.2f}秒）")
    print(f"判定率 decided_ratio = {metrics['decided_ratio']:.3f}")
    print(f"誤り {metrics['wrong_s']:.2f}秒、判定保留 {metrics['undecided_s']:.2f}秒、予測なし {metrics['missing_s']:.2f}秒")
else:
    print("正解注釈がないため、評価は行いません")

## 8. 結果を保存する

推定結果のCSV、区間ごとの類似度のCSV（N の区間は類似度を計算しないので空欄）、解析条件のJSON（設定、入力、ライブラリの版、コミット）、図（PNGとPDF）を保存します。

In [ ]:
extra = {"evaluation": metrics, "annotation": str(ANNOTATION_PATH)} if metrics else None
saved = save_analysis(analysis, OUTPUT_DIR, extra, figure, pdf=True)
for name, path in saved.items():
    print(f"{name:12} {path}")

# Colab で手元にダウンロードするときは、次の2行の先頭の # を消して実行します
# from google.colab import files
# files.download(str(saved["chords_csv"]))

## 9. 設定を変えて比べる（練習）

次の関数は、設定を1つだけ変えながら解析し、結果を表にします。`replace(settings, 名前=値)` は、設定の一部だけを変えたコピーを作ります。

In [ ]:
def compare_settings(name, values):
    """設定 name を values の値に1つずつ変えて解析し、結果を表にして表示します。"""
    print(f"{name:>20} | 区間の数 | 判定保留[秒] | 正解率 | 判定率")
    for value in values:
        trial = replace(settings, **{name: value})
        result = recognise_chords(clip, trial)
        undecided_s = sum(s.end_s - s.start_s for s in result.segments if s.status == "undecided")
        accuracy, decided = "-", "-"
        if reference:
            scores = evaluate_segments(reference, result.segments)
            accuracy, decided = f"{scores['accuracy']:.3f}", f"{scores['decided_ratio']:.3f}"
        print(f"{str(value):>20} | {len(result.segments):>8} | {undecided_s:>12.2f} | {accuracy:>6} | {decided:>6}")

### 練習1：時間幅 `segment_seconds` を変える

時間幅を 0.25、0.5、1.0、2.0 秒に変えると、区間の数・判定保留・正解率はどう変わるでしょうか。実行する前に予想してから確かめましょう。

In [ ]:
compare_settings("segment_seconds", [0.25, 0.5, 1.0, 2.0])

### 練習2：判定のしきい値 `min_margin`・`min_score` を変える

しきい値を大きくすると、判定保留・誤り・正解率・判定率はそれぞれどう変わるでしょうか。

In [ ]:
compare_settings("min_margin", [0.0, 0.01, 0.03, 0.05])
print()
compare_settings("min_score", [0.5, 0.6, 0.7, 0.8])

### 練習3：平滑化 `smoothing_segments` を変える

平滑化を強くすると、コードの区間の数と、コードが変わる位置の付近の結果はどう変わるでしょうか。図も描いて確かめましょう。

In [ ]:
compare_settings("smoothing_segments", [1, 3, 5, 9])

strong = recognise_chords(clip, replace(settings, smoothing_segments=9))
fig = plot_analysis(strong, "smoothing_segments = 9", reference)
plt.show()

### 練習4（発展）：入力や関数を変える

- `SOURCE = "demo"` にして合成デモで試し、実際の録音との違いを比べましょう。
- `chroma_method` を `"stft"` に変えて `compare_settings("chroma_method", ["cqt", "stft"])` を実行しましょう。
- `src/guitar_chord_lab/chords.py` の `make_chord_template` を変えて（例：根音だけ 1.5 にする）、結果を比べましょう。関数を変えたら、ランタイムを再起動してからセットアップのセルを実行し直します。

---

## 期待される観察と解答例

結果の数値は、入力の音源や切り出し位置で変わります。ここでは既定の入力（GuitarSet の `01_Rock3-117-Bb_comp` の先頭20秒）で見られる傾向を説明します。

**練習1（時間幅）**
- 時間幅を短くすると区間が細かくなり、コードが変わる位置を細かく追えます。一方、1区間に含まれる音が少なくなるため、ストロークの合間などで結果が揺れやすくなります。
- 時間幅を長くすると結果は安定しますが、1つの区間に2つのコードが混ざりやすくなります。コードが約2秒ごとに変わるこの曲で2.0秒にすると、境界をまたぐ区間が増え、誤りや判定保留が増えます。
- 解答例：「時間幅は、コードが変わる間隔より十分短くする必要がある。短すぎても不安定になるので、0.25〜0.5秒程度が使いやすい」。

**練習2（しきい値）**
- `min_margin` や `min_score` を大きくすると、判定保留が増え、誤りの時間は減ります。判定保留は不正解として数えるため、正解率と判定率は下がります。
- 既定の入力では、`min_margin = 0.0` のとき正解率が最も高くなります。その代わり判定保留がなくなり、誤りの時間は増えます。既定値（0.01）は、正解率を最大にする値ではなく、決めきれない区間を「保留」として知らせるための値です。
- 解答例：「しきい値は『誤った候補を出す』か『判定を保留する』かの調整である。耳コピ支援では、保留された区間を人が聴いて確かめる使い方ができる。正解率だけを見て選ばず、判定率も一緒に確認する」。

**練習3（平滑化）**
- 平滑化を強くすると、短い揺れが減って区間の数が減ります。強くしすぎると、コードが変わる位置がぼやけ、短いコードが前後のコードに吸収されたり、境界付近が判定保留になったりします。
- 解答例：「平滑化の幅（区間数 × 時間幅）が、1つのコードが続く時間より長くなると逆効果になる」。

**練習4**
- 合成デモは倍音の構造が単純で、正解率が実録音より高くなりやすいです。合成音で成功しても、実際のギター録音での精度の根拠にはなりません。
- STFT のクロマは、低い音の周波数の分解能が CQT より粗いため、結果が変わります。既定の入力や、既定値を選ぶときに使った調整用の録音では CQT のほうが正解率が高くなりましたが、素材によって異なる可能性があります。設定は調整用の素材で比べてから決めます。